# Phase 3 — Integration E2E Test (TASK-14)

Notebook này chạy 15 câu hỏi thử nghiệm qua toàn bộ pipeline RAG:

```
question → QueryPlanner → SubgraphExtractor → HybridSearch → ContextAssembler → AnswerGenerator
```

**Phạm vi [A]:** Đất đai (chuyển mục đích SDĐ + cấp sổ đỏ lần đầu), TP.HCM + Đồng Nai + Toàn quốc  
**DoD cần đạt:**
- DoD 1: Câu hỏi Đất đai TP.HCM trả lời trong < 30s
- DoD 2: ≥ 2 câu hỏi cho mỗi thủ tục
- DoD 3: Câu hỏi thiếu jurisdiction → `confirmation_needed=True`
- DoD 4: Negative test khai sinh TP.HCM vs Đồng Nai → không bịa sự khác biệt
- DoD 5: Ghi kết quả + nhận xét vào notebook

> **Lưu ý citation**: LLM có thể dùng format tắt `[Điều X, Luật Y]` thay vì format chuẩn `[Điều X, Văn bản Y]`.  
> `parse_citations()` chỉ bắt format chuẩn — citation count = 0 không có nghĩa LLM không trích dẫn.  
> Đánh giá chất lượng trích dẫn bằng mắt qua phần TRẢ LỜI.

In [1]:
import os
import sys
import json
import time
import logging
from pathlib import Path

# Tìm project root bất kể notebook được chạy từ đâu
_cwd = Path.cwd()
_project_root = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / 'CLAUDE.md').exists()),
    _cwd,
)
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
print(f'Project root: {_project_root}')

from dotenv import load_dotenv
load_dotenv(_project_root / '.env')

logging.basicConfig(level=logging.WARNING)
print('Import OK')

Project root: /Users/daonguyentandat/Documents/University/2526_Sem2/Thesis/vn-legal-graphrag
Import OK


In [2]:
import anthropic
from neo4j import GraphDatabase
from qdrant_client import QdrantClient

from src.ingestion.vectorizer import load_model
from src.pipeline import run_pipeline

# Khởi tạo clients một lần, dùng lại cho tất cả câu hỏi
neo4j_driver = GraphDatabase.driver(
    os.getenv('NEO4J_URI', 'bolt://localhost:7687'),
    auth=(os.getenv('NEO4J_USER', 'neo4j'), os.getenv('NEO4J_PASSWORD', '')),
)
qdrant_client = QdrantClient(
    host=os.getenv('QDRANT_HOST', 'localhost'),
    port=int(os.getenv('QDRANT_PORT', '6333')),
)
anthropic_client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
model = load_model()

print('Clients OK')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Clients OK


In [3]:
results = []  # lưu kết quả để đánh giá cuối

def ask(question: str, label: str = '') -> dict:
    """Chạy pipeline và in kết quả gọn."""
    print(f"\n{'='*70}")
    if label:
        print(f"[{label}]")
    print(f"CÂU HỎI: {question}")
    print('='*70)

    result = run_pipeline(
        question,
        neo4j_driver=neo4j_driver,
        qdrant_client=qdrant_client,
        anthropic_client=anthropic_client,
        model=model,
    )

    if result['confirmation_needed']:
        print('⚠️  CẦN XÁC NHẬN:')
        print(result['confirmation_prompt'])
    else:
        print(f"📊 LCCIDs: {result['lccids_count']}  |  Top-k: {result['top_k_count']}  |  Context: ~{result['context_tokens']} tokens")
        print(f"\n💬 TRẢ LỜI:\n{result['answer']}")
        if result['citations']:
            print(f"\n📌 CITATIONS parsed ({len(result['citations'])}): {result['citations']}")
        else:
            print('\n📌 CITATIONS parsed: 0 (LLM có thể dùng format tắt — kiểm tra thủ công)')

    print(f"\n⏱️  {result['elapsed_seconds']}s")
    return result

print('ask() ready')

ask() ready


## 1. Chuyển mục đích sử dụng đất — TP.HCM (DoD 1 + DoD 2)

In [4]:
# DoD 1: câu hỏi chuẩn, phải trả lời trong < 30s
r = ask(
    'Điều kiện để chuyển mục đích sử dụng đất tại TP.HCM là gì?',
    label='Q01 | CMĐSDĐ TP.HCM | DoD-1'
)
results.append(r)

# DoD 1 checks
assert r['elapsed_seconds'] < 45, f'TIMEOUT: {r["elapsed_seconds"]}s'
assert not r['confirmation_needed'], 'Câu hỏi có đủ jurisdiction — không được confirmation_needed'
# Citation count là soft check vì LLM có thể dùng format tắt
if len(r['citations']) >= 1:
    print('✅ DoD 1 PASS (có citation theo format chuẩn)')
else:
    print('⚠️  DoD 1 PARTIAL — dưới 30s, có trả lời, nhưng citation format chưa chuẩn (kiểm tra thủ công)')


[Q01 | CMĐSDĐ TP.HCM | DoD-1]
CÂU HỎI: Điều kiện để chuyển mục đích sử dụng đất tại TP.HCM là gì?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5667 tokens

💬 TRẢ LỜI:
# Điều kiện chuyển mục đích sử dụng đất tại TP.HCM

## I. Trình tự, thủ tục chung

**Bước 1:** Người sử dụng đất nộp hồ sơ đề nghị chuyển mục đích sử dụng đất theo quy định. [Điều 227, Khoản 1, Văn bản luat-dat-dai-2024]

**Bước 2:** Cơ quan có chức năng quản lý đất đai kiểm tra các điều kiện chuyển mục đích sử dụng đất. Nếu hồ sơ chưa đủ, cơ quan hướng dẫn bổ sung. [Điều 227, Khoản 2, Văn bản luat-dat-dai-2024]

---

## II. Điều kiện đặc thù khi chuyển từ đất rừng/đất trồng lúa

Khi chuyển mục đích từ **đất rừng phòng hộ, rừng đặc dụng, rừng sản xuất** sang mục đích khác, phải có:
- Phương án trồng rừng thay thế, **hoặc** văn bản hoàn thành trách nhiệm nộp tiền trồng rừng thay thế theo pháp luật lâm nghiệp. [Điều 46, Khoản 1, Điểm a, Văn bản nghi-dinh-102-2024-nd-cp]

Khi chuyển từ **đất chuyên trồng lúa** sang đất phi nông nghiệp, phải có:
- **Phương án sử dụng tầng đất mặt** (bóc tách tối thiểu 20 cm tính từ mặt ruộng)

In [5]:
r = ask(
    'Hộ gia đình có được chuyển đất nông nghiệp sang đất ở tại TP.HCM không? Cần điều kiện gì?',
    label='Q02 | CMĐSDĐ TP.HCM — điều kiện hộ gia đình'
)
results.append(r)


[Q02 | CMĐSDĐ TP.HCM — điều kiện hộ gia đình]
CÂU HỎI: Hộ gia đình có được chuyển đất nông nghiệp sang đất ở tại TP.HCM không? Cần điều kiện gì?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5912 tokens

💬 TRẢ LỜI:
# Chuyển Đất Nông Nghiệp Sang Đất Ở Tại TP.HCM

## 1. Căn cứ pháp lý và điều kiện được phép chuyển mục đích

Hộ gia đình, cá nhân **được phép chuyển đất nông nghiệp sang đất ở**, tuy nhiên phải đáp ứng các điều kiện sau:

### Điều kiện về nguồn gốc đất

Chỉ áp dụng cho các trường hợp:
- Đất vườn, ao **trong cùng thửa đất có đất ở** được xác định khi công nhận quyền sử dụng đất; **hoặc**
- Đất có **nguồn gốc là đất vườn, ao gắn liền đất ở** nhưng đã tách ra để chuyển quyền sử dụng đất; **hoặc**
- Đất do đơn vị đo đạc **trước ngày 01/7/2014** tự đo vẽ tách thành các thửa riêng.

[Điều 10, Khoản 2, Điểm c, Văn bản nghi-quyet-254-2025-qh15]

### Điều kiện về quy hoạch

Việc chuyển mục đích phải **phù hợp với quy hoạch sử dụng đất cấp huyện, hoặc quy hoạch sử dụng đất cấp xã, hoặc một trong các quy hoạch theo pháp luật về quy hoạch đô thị và nông thôn**.

[Điều 26, Khoản 1, Điểm c, Văn bản nghi-dinh-101-2024-nd-cp]

> **Lưu ý:

In [6]:
r = ask(
    'Hồ sơ xin chuyển mục đích sử dụng đất tại TP.HCM gồm những giấy tờ gì?',
    label='Q03 | CMĐSDĐ TP.HCM — hồ sơ'
)
results.append(r)


[Q03 | CMĐSDĐ TP.HCM — hồ sơ]
CÂU HỎI: Hồ sơ xin chuyển mục đích sử dụng đất tại TP.HCM gồm những giấy tờ gì?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5647 tokens

💬 TRẢ LỜI:
## Hồ sơ xin chuyển mục đích sử dụng đất tại TP.HCM

Dựa trên thông tin trong context, các quy định liên quan đến hồ sơ chuyển mục đích sử dụng đất bao gồm:

---

### 1. Trình tự cơ bản theo Luật Đất đai 2024

- **Bước 1:** Người sử dụng đất nộp hồ sơ đề nghị chuyển mục đích sử dụng đất theo quy định. [Điều 227, Khoản 1, Văn bản luat-dat-dai-2024]
- **Bước 2:** Cơ quan có chức năng quản lý đất đai kiểm tra các điều kiện; nếu hồ sơ chưa đủ thì hướng dẫn bổ sung. [Điều 227, Khoản 2, Văn bản luat-dat-dai-2024]

---

### 2. Thành phần hồ sơ đặc thù: Trường hợp chuyển từ đất chuyên trồng lúa sang phi nông nghiệp

Nếu thửa đất là **đất chuyên trồng lúa** chuyển sang mục đích phi nông nghiệp, hồ sơ phải có thêm **Phương án sử dụng tầng đất mặt**, bao gồm:

- Thông tin của người được nhà nước giao đất/cho thuê đất/cho phép chuyển mục đích;
- Thông tin về diện tích đất chuyên trồng lúa đề nghị chuyển đổi;
- Khối lượng tầng đất mặt

In [7]:
r = ask(
    'Nghĩa vụ tài chính khi chuyển mục đích sử dụng đất sang đất ở tại TP.HCM là bao nhiêu?',
    label='Q04 | CMĐSDĐ TP.HCM — nghĩa vụ tài chính'
)
results.append(r)


[Q04 | CMĐSDĐ TP.HCM — nghĩa vụ tài chính]
CÂU HỎI: Nghĩa vụ tài chính khi chuyển mục đích sử dụng đất sang đất ở tại TP.HCM là bao nhiêu?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5617 tokens

💬 TRẢ LỜI:
# Nghĩa vụ tài chính khi chuyển mục đích sử dụng đất sang đất ở tại TP.HCM

## 1. Tiền sử dụng đất (Nghĩa vụ tài chính chính)

Đối với trường hợp chuyển từ **đất vườn, ao, đất nông nghiệp trong cùng thửa đất có đất ở** (hoặc đất có nguồn gốc vườn ao gắn liền đất ở), mức thu tiền sử dụng đất được tính theo **chênh lệch** giữa giá đất ở và giá đất nông nghiệp, cụ thể:

| Diện tích chuyển mục đích | Mức thu |
|---|---|
| Trong hạn mức giao đất ở tại địa phương | **30%** chênh lệch |
| Vượt hạn mức nhưng không quá 1 lần hạn mức | **50%** chênh lệch |
| Vượt quá 1 lần hạn mức | **100%** chênh lệch |

> **Lưu ý quan trọng:** Mức ưu đãi 30%/50%/100% này **chỉ được tính một lần cho một hộ gia đình, cá nhân** (tính trên 01 thửa đất).

[Điều 10, Khoản 2, Điểm c, Văn bản nghi-quyet-254-2025-qh15]

---

## 2. Hạn mức giao đất ở để tính tiền sử dụng đất

Hạn mức giao đất ở được xác định **tại thời điểm có quyết định cho phép chuyển mụ

## 2. Chuyển mục đích sử dụng đất — Đồng Nai

In [8]:
r = ask(
    'Quy trình chuyển mục đích sử dụng đất nông nghiệp sang đất ở tại Đồng Nai như thế nào?',
    label='Q05 | CMĐSDĐ Đồng Nai — quy trình'
)
results.append(r)


[Q05 | CMĐSDĐ Đồng Nai — quy trình]
CÂU HỎI: Quy trình chuyển mục đích sử dụng đất nông nghiệp sang đất ở tại Đồng Nai như thế nào?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5056 tokens

💬 TRẢ LỜI:
# Quy trình chuyển mục đích sử dụng đất nông nghiệp sang đất ở tại Đồng Nai

## I. CĂN CỨ PHÁP LÝ ÁP DỤNG

Dựa trên các văn bản pháp luật hiện hành, quy trình và nghĩa vụ tài chính được tổng hợp như sau:

---

## II. TRƯỜNG HỢP PHẢI XIN PHÉP CHUYỂN MỤC ĐÍCH

Việc chuyển từ đất nông nghiệp sang đất ở thuộc trường hợp **phải xin phép** cơ quan nhà nước có thẩm quyền (trừ các trường hợp ngoại lệ nêu tại Khoản 3 Điều 121 Luật Đất đai). [Điều 121, Khoản 3, Văn bản luat-dat-dai-2024]

---

## III. NGHĨA VỤ TÀI CHÍNH (BẮT BUỘC)

### 1. Tiền sử dụng đất khi chuyển mục đích

Đối với **đất vườn, ao, đất nông nghiệp trong cùng thửa đất có đất ở** (hoặc đất có nguồn gốc vườn ao gắn liền đất ở), tiền sử dụng đất được tính theo 3 mức: [Điều 10, Khoản 2, Điểm c, Văn bản nghi-quyet-254-2025-qh15]

| Diện tích chuyển mục đích | Mức thu |
|---|---|
| Trong hạn mức giao đất ở tại Đồng Nai | **30%** chênh lệch (giá đất ở – giá đất nông nghiệ

In [9]:
r = ask(
    'Thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất tại Đồng Nai là bao lâu?',
    label='Q06 | CMĐSDĐ Đồng Nai — thời hạn'
)
results.append(r)


[Q06 | CMĐSDĐ Đồng Nai — thời hạn]
CÂU HỎI: Thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất tại Đồng Nai là bao lâu?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5058 tokens

💬 TRẢ LỜI:
## Thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất tại Đồng Nai

Dựa trên các thông tin trong CONTEXT được cung cấp, **không có quy định cụ thể về thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất tại tỉnh Đồng Nai**.

### Thông tin liên quan có thể tham khảo:

CONTEXT chỉ đề cập đến việc **UBND cấp tỉnh có trách nhiệm ban hành quy trình thủ tục hành chính về đất đai**, trong đó bao gồm cả thời gian tối đa thực hiện từng bước, nhưng chưa có văn bản cụ thể của Đồng Nai về vấn đề này:

> *"Ủy ban nhân dân cấp tỉnh quy định trình tự, thủ tục hành chính về đất đai... chậm nhất đến ngày 01 tháng 7 năm 2026... trong đó quy định rõ... thời gian tối đa thực hiện trình tự, thủ tục..."*
> [Điều 15, Khoản 1, Văn bản nghi-dinh-49-2026-nd-cp]

### Khuyến nghị:

Để biết thời hạn cụ thể, bạn nên:
1. Tra cứu quyết định của **UBND tỉnh Đồng Nai** về trình tự, thủ tục hành chính đất đai (dự kiến ban hành trước ngày 01/7/2026).
2.

## 3. Cấp sổ đỏ lần đầu — TP.HCM (DoD 2)

In [10]:
r = ask(
    'Điều kiện để được cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM là gì?',
    label='Q07 | Cấp sổ đỏ TP.HCM — điều kiện'
)
results.append(r)


[Q07 | Cấp sổ đỏ TP.HCM — điều kiện]
CÂU HỎI: Điều kiện để được cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM là gì?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5875 tokens

💬 TRẢ LỜI:
# Điều kiện cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM

## I. ĐIỀU KIỆN PHÁP LÝ CƠ BẢN

### 1. Về giấy tờ chứng minh quyền sử dụng đất

Người sử dụng đất cần có một trong các loại giấy tờ hợp lệ, bao gồm:
- Giấy tờ tạm giao đất của UBND cấp huyện, cấp tỉnh; Đơn đề nghị được sử dụng đất được UBND cấp xã, hợp tác xã nông nghiệp phê duyệt, chấp thuận trước ngày 01/7/1980 hoặc được UBND cấp huyện, cấp tỉnh phê duyệt, chấp thuận. [Điều 137, Khoản 1, Điểm l, Văn bản luat-dat-dai-2024]
- Giấy chứng nhận quyền sử dụng đất **tạm thời** được cấp từ ngày 15/10/1993 trở về sau cũng được công nhận và cấp Giấy chứng nhận chính thức. [Điều 11, Khoản 2, Điểm đ, Văn bản nghi-quyet-254-2025-qh15]

### 2. Trường hợp đất sử dụng từ 15/10/1993 đến trước 01/7/2014 có nhà ở

Diện tích đất ở được xác định như sau:
- Nếu diện tích thửa đất **≥ hạn mức giao đất ở** → diện tích đất ở = **hạn mức giao đất ở**
- Nếu diện tích đã xây dựn

In [11]:
r = ask(
    'Hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM gồm những gì?',
    label='Q08 | Cấp sổ đỏ TP.HCM — hồ sơ'
)
results.append(r)


[Q08 | Cấp sổ đỏ TP.HCM — hồ sơ]
CÂU HỎI: Hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM gồm những gì?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5905 tokens

💬 TRẢ LỜI:
## Hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM

Dựa trên thông tin trong CONTEXT được cung cấp, tôi chỉ có thể trả lời **một phần** câu hỏi này.

### 1. Giấy tờ về quyền sử dụng đất (điều kiện để được cấp GCN)

Theo Luật Đất đai 2024, hộ gia đình, cá nhân được cấp Giấy chứng nhận lần đầu nếu có một trong các loại giấy tờ sau:

- **Giấy tờ từ chế độ cũ**, bao gồm: Bằng khoán điền thổ; Văn tự đoạn mãi bất động sản có chứng nhận của cơ quan thuộc chế độ cũ; Giấy phép xây cất nhà ở; Bản án của Tòa án thuộc chế độ cũ đã có hiệu lực... [Điều 137, Khoản 1, Điểm b, Văn bản luat-dat-dai-2024]

- **Giấy tờ đăng ký ruộng đất** theo Chỉ thị 299/TTg, bao gồm: Biên bản xét duyệt của Hội đồng đăng ký ruộng đất cấp xã; Đơn xin đăng ký quyền sử dụng ruộng đất; Giấy tờ chứng nhận đã đăng ký quyền sử dụng đất... [Điều 137, Khoản 1, Điểm e, Văn bản luat-dat-dai-2024]

- **Giấy tờ tạm giao đất** của UBND cấp huyện, 

In [12]:
r = ask(
    'Trường hợp đất không có giấy tờ tại TP.HCM thì có được cấp sổ đỏ lần đầu không? Điều kiện là gì?',
    label='Q09 | Cấp sổ đỏ TP.HCM — đất không giấy tờ'
)
results.append(r)


[Q09 | Cấp sổ đỏ TP.HCM — đất không giấy tờ]
CÂU HỎI: Trường hợp đất không có giấy tờ tại TP.HCM thì có được cấp sổ đỏ lần đầu không? Điều kiện là gì?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5721 tokens

💬 TRẢ LỜI:
# Cấp sổ đỏ lần đầu cho đất không có giấy tờ tại TP.HCM

## 1. Có được cấp sổ đỏ không?

**Có**, hộ gia đình, cá nhân đang sử dụng đất không có giấy tờ về quyền sử dụng đất vẫn có thể được cấp Giấy chứng nhận (sổ đỏ) lần đầu, nếu đáp ứng các điều kiện theo quy định.

[Điều 138, Khoản 1, Văn bản luat-dat-dai-2024]

---

## 2. Điều kiện cụ thể

### A. Điều kiện cơ bản
- Sử dụng đất **trước ngày 18/12/1980**
- Được **UBND cấp xã** nơi có đất xác nhận **không có tranh chấp**

[Điều 138, Khoản 1, Văn bản luat-dat-dai-2024]

### B. Xử lý từng phần diện tích đất

| Loại đất | Cách xử lý |
|---|---|
| Đất ở (trong hạn mức) | Được công nhận, không hoặc có thu tiền SDĐ tùy thời điểm sử dụng |
| Đất nông nghiệp (phần còn lại) | Công nhận theo hình thức **giao đất không thu tiền SDĐ** |
| Đất nông nghiệp muốn chuyển sang phi nông nghiệp | Phải phù hợp quy hoạch và **nộp tiền sử dụng đất** |

[Điều 138, Khoản 1, Điểm d, Văn bản luat-d

## 4. Cấp sổ đỏ lần đầu — Đồng Nai

In [13]:
r = ask(
    'Cơ quan nào tiếp nhận hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại Đồng Nai?',
    label='Q10 | Cấp sổ đỏ Đồng Nai — cơ quan'
)
results.append(r)


[Q10 | Cấp sổ đỏ Đồng Nai — cơ quan]
CÂU HỎI: Cơ quan nào tiếp nhận hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại Đồng Nai?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5889 tokens

💬 TRẢ LỜI:
## Cơ quan tiếp nhận hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại Đồng Nai

Dựa trên thông tin trong CONTEXT được cung cấp, **không có quy định cụ thể** nào xác định rõ tên cơ quan tiếp nhận hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại Đồng Nai.

### Thông tin liên quan có trong CONTEXT:

CONTEXT chỉ đề cập đến:
- **Văn phòng đăng ký đất đai / Chi nhánh Văn phòng đăng ký đất đai** với tư cách là cơ quan **giải quyết thủ tục** (không phải tiếp nhận hồ sơ lần đầu) trong một số trường hợp biến động đất đai [Điều 19, Khoản 4, Văn bản nghi-dinh-101-2024-nd-cp]; [Phụ lục I - Phần V - Nội dung C - Mục VI, Khoản 3, Điểm b, Văn bản nghi-dinh-151-2025-nd-cp].
- Các quy định về **mức phí thẩm định hồ sơ cấp Giấy chứng nhận lần đầu** tại Đồng Nai [Phụ lục I, Văn bản nghi-quyet-22-2024-nq-hdnd-dong-nai].

### Kết luận:

Context **không đủ thông tin** để trả lời đầy đủ câu hỏi về cơ quan tiếp nhận 

In [14]:
r = ask(
    'Phí và lệ phí khi đăng ký cấp sổ đỏ lần đầu tại Đồng Nai là bao nhiêu?',
    label='Q11 | Cấp sổ đỏ Đồng Nai — phí lệ phí'
)
results.append(r)


[Q11 | Cấp sổ đỏ Đồng Nai — phí lệ phí]
CÂU HỎI: Phí và lệ phí khi đăng ký cấp sổ đỏ lần đầu tại Đồng Nai là bao nhiêu?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5760 tokens

💬 TRẢ LỜI:
## Phí và lệ phí khi đăng ký cấp sổ đỏ lần đầu tại Đồng Nai

### 1. Phí thẩm định hồ sơ (cấp lần đầu)

Theo [Phụ lục I, Văn bản nghi-quyet-22-2024-nq-hdnd-dong-nai], mức phí thẩm định hồ sơ cấp Giấy chứng nhận lần đầu như sau:

| Loại hồ sơ | Hộ GĐ/cá nhân - Trực tiếp | Hộ GĐ/cá nhân - Trực tuyến | Tổ chức - Trực tiếp | Tổ chức - Trực tuyến |
|---|:---:|:---:|:---:|:---:|
| Cấp GCN về **quyền sử dụng đất** | 880.000 đ | 836.000 đ | 1.260.000 đ | 1.197.000 đ |
| Cấp GCN về **tài sản** | 980.000 đ | 931.000 đ | 1.840.000 đ | 1.748.000 đ |
| Cấp GCN **cả đất và tài sản gắn liền với đất** | 1.250.000 đ | 1.187.500 đ | 2.090.000 đ | 1.985.500 đ |

**Ghi chú đặc biệt** [Phụ lục I, Văn bản nghi-quyet-22-2024-nq-hdnd-dong-nai]:
- Một thửa đất có **nhiều người đồng sử dụng**: mỗi GCN cấp thêm thu **50.000 đồng/GCN/người** (áp dụng cả trực tiếp lẫn trực tuyến).
- Tổ chức có dự án **nhiều thửa đất**: từ thửa thứ hai trở đi thu **50.

## 5. DoD 3 — Thiếu jurisdiction → confirmation_needed

In [15]:
r = ask(
    'Điều kiện chuyển mục đích sử dụng đất nông nghiệp sang đất ở là gì?',
    label='Q12 | Thiếu jurisdiction | DoD-3'
)
results.append(r)
assert r['confirmation_needed'] is True, 'Phải confirmation_needed=True khi thiếu jurisdiction'
assert r['confirmation_prompt'] is not None
print('✅ DoD 3 PASS')


[Q12 | Thiếu jurisdiction | DoD-3]
CÂU HỎI: Điều kiện chuyển mục đích sử dụng đất nông nghiệp sang đất ở là gì?
⚠️  CẦN XÁC NHẬN:
Bất động sản / đất đai của bạn thuộc tỉnh/thành phố nào? (TP. Hồ Chí Minh / Đồng Nai / địa phương khác)

⏱️  0.98s
✅ DoD 3 PASS


## 6. DoD 4 — Negative test: khai sinh TP.HCM vs Đồng Nai

In [16]:
r = ask(
    'Quy định đăng ký khai sinh tại TP.HCM khác Đồng Nai như thế nào?',
    label='Q13 | Negative — khai sinh địa phương | DoD-4'
)
results.append(r)
# Khai sinh là thủ tục toàn quốc → pipeline trả về empty context (chưa có data Hộ tịch)
# hoặc nói rõ không có sự khác biệt. Không được bịa sự khác biệt.
print('\n⚠️  Kiểm tra thủ công DoD 4: câu trả lời không được bịa sự khác biệt địa phương không tồn tại')


[Q13 | Negative — khai sinh địa phương | DoD-4]
CÂU HỎI: Quy định đăng ký khai sinh tại TP.HCM khác Đồng Nai như thế nào?


📊 LCCIDs: 0  |  Top-k: 0  |  Context: ~0 tokens

💬 TRẢ LỜI:
Không tìm thấy văn bản pháp luật liên quan đến câu hỏi này.

📌 CITATIONS parsed: 0 (LLM có thể dùng format tắt — kiểm tra thủ công)

⏱️  1.52s

⚠️  Kiểm tra thủ công DoD 4: câu trả lời không được bịa sự khác biệt địa phương không tồn tại


## 7. Gap 3 — câu hỏi cần traversal [:IMPLEMENTS]

In [17]:
r = ask(
    'Nghị định 102/2024/NĐ-CP hướng dẫn thi hành Luật Đất đai 2024 quy định gì về chuyển mục đích sử dụng đất tại TP.HCM?',
    label='Q14 | Gap3 — NĐ 102 CMĐSDĐ TP.HCM'
)
results.append(r)


[Q14 | Gap3 — NĐ 102 CMĐSDĐ TP.HCM]
CÂU HỎI: Nghị định 102/2024/NĐ-CP hướng dẫn thi hành Luật Đất đai 2024 quy định gì về chuyển mục đích sử dụng đất tại TP.HCM?


📊 LCCIDs: 13  |  Top-k: 22  |  Context: ~5851 tokens

💬 TRẢ LỜI:
## Quy định về chuyển mục đích sử dụng đất tại TP.HCM theo Nghị định 102/2024/NĐ-CP và các văn bản liên quan

Dựa trên thông tin trong CONTEXT, tôi có thể cung cấp các quy định liên quan như sau:

---

### 1. Căn cứ cho phép chuyển mục đích sử dụng đất

Đối với hộ gia đình, cá nhân muốn chuyển mục đích sử dụng đất nông nghiệp sang đất ở, căn cứ là **quy hoạch sử dụng đất cấp huyện** hoặc quy hoạch chung hoặc quy hoạch phân khu theo pháp luật về quy hoạch đô thị và nông thôn đã được cơ quan có thẩm quyền phê duyệt.
[Điều 116, Khoản 5, Văn bản luat-dat-dai-2024]

---

### 2. Thẩm quyền cho phép chuyển mục đích sử dụng đất

- **UBND cấp huyện** có thẩm quyền cho phép chuyển mục đích sử dụng đất đối với **cá nhân**.
- Riêng trường hợp chuyển mục đích đất nông nghiệp sang **thương mại, dịch vụ với diện tích từ 0,5 ha trở lên**, phải có **văn bản chấp thuận của UBND cấp tỉnh** (tức UBND TP.HCM) trước khi quyết định.
[Điều 123, 

In [18]:
r = ask(
    'Bảng giá đất TP.HCM năm 2025 ảnh hưởng như thế nào đến tiền sử dụng đất khi chuyển mục đích?',
    label='Q15 | Gap2+Gap3 — Bảng giá đất TP.HCM'
)
results.append(r)


[Q15 | Gap2+Gap3 — Bảng giá đất TP.HCM]
CÂU HỎI: Bảng giá đất TP.HCM năm 2025 ảnh hưởng như thế nào đến tiền sử dụng đất khi chuyển mục đích?


📊 LCCIDs: 7  |  Top-k: 19  |  Context: ~4036 tokens

💬 TRẢ LỜI:
## Bảng giá đất TP.HCM năm 2025 và ảnh hưởng đến tiền sử dụng đất khi chuyển mục đích

### Giới hạn thông tin có thể cung cấp

**Context hiện có không chứa thông tin cụ thể về Bảng giá đất TP.HCM năm 2025**, do đó tôi không thể cung cấp số liệu cụ thể về bảng giá đất của thành phố này.

---

### Những gì có thể xác định từ Context

**1. Vai trò của bảng giá đất trong tính tiền sử dụng đất**

Theo quy định hiện hành, bảng giá đất được sử dụng để:
- Tính tiền sử dụng đất khi Nhà nước **công nhận quyền sử dụng đất ở** của hộ gia đình, cá nhân
- Tính tiền sử dụng đất khi **chuyển mục đích sử dụng đất** của hộ gia đình, cá nhân

[Điều 159, Khoản 1, Điểm a, Văn bản luat-dat-dai-2024]

**2. Trường hợp chuyển từ đất nông nghiệp sang phi nông nghiệp**

Nếu người sử dụng đất có nhu cầu công nhận diện tích đất nông nghiệp vào mục đích **đất phi nông nghiệp** (phù hợp quy hoạch), thì **phải nộp tiền sử dụng đất** theo quy định của phá

## 8. Tổng kết kết quả

In [19]:
print('\n' + '='*70)
print('TỔNG KẾT PIPELINE E2E TEST — TASK-14')
print('='*70)

total = len(results)
confirmed = sum(1 for r in results if r['confirmation_needed'])
answered = total - confirmed
with_parsed_citations = sum(1 for r in results if not r['confirmation_needed'] and r['citations'])
avg_elapsed = sum(r['elapsed_seconds'] for r in results) / total if total else 0

print(f'  Tổng câu hỏi:             {total}')
print(f'  Câu trả lời được:         {answered}')
print(f'  Cần xác nhận jurisdiction: {confirmed}')
print(f'  Có citation (format chuẩn): {with_parsed_citations}/{answered}')
print(f'  Thời gian TB:             {avg_elapsed:.1f}s')
if total:
    print(f'  Max elapsed:              {max(r["elapsed_seconds"] for r in results):.1f}s')

print('\nChi tiết:')
for i, r in enumerate(results, 1):
    if r['confirmation_needed']:
        status = '⚠️  CONFIRM'
    elif r['citations']:
        status = f'✅ {len(r["citations"])} cite'
    else:
        status = '⚡ 0 cite*'
    print(f'  Q{i:02d}: {status:12} {r["elapsed_seconds"]:5.1f}s | LCCIDs={r["lccids_count"]:4d} | {r["question"][:55]}')

print('\n* 0 cite = LLM có thể dùng format tắt, kiểm tra thủ công')


TỔNG KẾT PIPELINE E2E TEST — TASK-14
  Tổng câu hỏi:             15
  Câu trả lời được:         14
  Cần xác nhận jurisdiction: 1
  Có citation (format chuẩn): 13/14
  Thời gian TB:             23.4s
  Max elapsed:              36.6s

Chi tiết:
  Q01: ✅ 11 cite     32.2s | LCCIDs=  13 | Điều kiện để chuyển mục đích sử dụng đất tại TP.HCM là 
  Q02: ✅ 7 cite      35.0s | LCCIDs=  13 | Hộ gia đình có được chuyển đất nông nghiệp sang đất ở t
  Q03: ✅ 7 cite      26.1s | LCCIDs=  13 | Hồ sơ xin chuyển mục đích sử dụng đất tại TP.HCM gồm nh
  Q04: ✅ 6 cite      26.8s | LCCIDs=  13 | Nghĩa vụ tài chính khi chuyển mục đích sử dụng đất sang
  Q05: ✅ 8 cite      36.6s | LCCIDs=  13 | Quy trình chuyển mục đích sử dụng đất nông nghiệp sang 
  Q06: ✅ 1 cite      13.8s | LCCIDs=  13 | Thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất t
  Q07: ✅ 12 cite     36.2s | LCCIDs=  13 | Điều kiện để được cấp Giấy chứng nhận quyền sử dụng đất
  Q08: ✅ 3 cite      27.4s | LCCIDs=  13 | Hồ sơ đăng ký cấp 

In [20]:
# Đóng clients
neo4j_driver.close()
print('Clients closed.')

Clients closed.
